<a href="https://colab.research.google.com/github/aabadmo4/DGT_Incidencias/blob/main/DGT_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

⚙️🎙️Instalación del motor de voz

In [ ]:

# Instalar pydub y la herramienta del sistema ffmpeg para manipular audio
!apt-get update -y -q
!apt-get install -y -q ffmpeg
!pip install -q pydub google-genai lxml requests

import os
import re
import requests
from collections import defaultdict
from IPython.display import Audio, display
from lxml import etree
from google.colab import userdata
from google import genai
from pydub import AudioSegment

print("✅ Entorno de audio avanzado configurado correctamente.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:3 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:6 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Fetched 3,917 B in 1s (3,752 B/s)
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
0 upgraded, 0 newly installed, 0 to remove and 53 not upgraded.
✅ Entorno de audio avanza

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


🧹📂 Extracción y limpieza de datos.

In [ ]:
import os
import re
import requests
from collections import defaultdict
from lxml import etree

def limpiar_y_extraer_detalles(record):
    traducciones_causa = {
        "ROADWORKS": "Obras",
        "ROADMAINTENANCE": "Mantenimiento",
        "CARRIAGEWAYCLOSURE": "Corte total de calzada",
        "ACCIDENT": "Accidente",
        "POORWEATHERCONDITIONS": "Meteorología adversa",
        "SNOW": "Nieve",
        "ICE": "Hielo",
        "FLOODING": "Inundación",
        "OBSTRUCTION": "Obstáculo",
        "TRAFFICCONGESTION": "Retención"
    }

    raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
    municipios, provincia, causas = [], "", []

    for t in raw_texts:
        t_clean = t.strip()
        if re.match(r'^\d{4}-\d{2}-\d{2}', t_clean): continue
        if re.match(r'^-?\d+\.\d+$', t_clean): continue
        if re.match(r'^[A-Z0-9_]{8,}$', t_clean): continue
        if t_clean.lower() in ['true', 'false', 'dgt', 'certain', 'active', 'segment', 'positive', 'mandatory', 'anyvehicle']: continue

        t_upper = t_clean.upper()
        if t_upper in traducciones_causa:
            causas.append(traducciones_causa[t_upper])
            continue

        if t_clean in ["Teruel", "Zaragoza", "Huesca", "Navarra", "La Rioja"]:
            provincia = t_clean
        elif len(t_clean) > 2 and not t_clean.replace('.', '').isdigit() and t_clean not in ["Aragón", "Comunidad Foral de Navarra"]:
            if t_clean not in municipios:
                municipios.append(t_clean)

    ubicacion_str = f"Entre/En: {', '.join(municipios)}" if municipios else "Tramo local"
    causa_str = f"Incidencia: {', '.join(set(causas))}" if causas else "Afección en la vía"

    return f"{provincia} | {ubicacion_str} | {causa_str}"

def obtener_incidencias_texto():
    url = "https://nap.dgt.es/datex2/v3/dgt/SituationPublication/datex2_v37.xml"
    headers = {'User-Agent': 'Mozilla/5.0'}

    regiones_mapa = {
        "ARAGÓN": ["ZARAGOZA", "HUESCA", "TERUEL", "ARAGON", "ARAGÓN"],
        "COMUNIDAD FORAL DE NAVARRA": ["NAVARRA", "PAMPLONA"],
        "LA RIOJA": ["RIOJA", "LOGROÑO"]
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        parser = etree.XMLParser(recover=True, encoding='utf-8')
        root = etree.fromstring(response.content, parser=parser)

        incidencias_por_zona = defaultdict(list)

        for record in root.xpath('//*[local-name()="situationRecord"]'):
            roads = record.xpath('.//*[local-name()="roadName"]/text()')
            road_name = roads[0].strip() if roads else "Vía local"

            raw_texts = [t.strip() for t in record.xpath('.//text()') if t.strip()]
            texto_evaluacion = f"{road_name} " + " ".join(raw_texts).upper()

            region_encontrada = None
            for region, terminos in regiones_mapa.items():
                if any(term in texto_evaluacion for term in terminos):
                    region_encontrada = region
                    break

            if region_encontrada:
                resumen = limpiar_y_extraer_detalles(record)
                detalle = f"- Código oficial: {road_name} -> {resumen}"
                incidencias_por_zona[region_encontrada].append(detalle)

        texto_resultado = ""
        for reg in ["ARAGÓN", "COMUNIDAD FORAL DE NAVARRA", "LA RIOJA"]:
            if reg in incidencias_por_zona:
                texto_resultado += f"\n--- REGIÓN: {reg} ---\n"
                texto_resultado += "\n".join(incidencias_por_zona[reg]) + "\n"

        return texto_resultado if texto_resultado else "Sin incidencias."

    except Exception as e:
        return f"Error extrayendo datos: {e}"

print("✅ Módulo XML cargado correctamente.")

✅ Módulo XML cargado correctamente.


🎙️📢Generación del boletín con Gemini y síntesis de voz mediante pyttsx3

In [ ]:
import datetime
import re
import requests
from google.colab import userdata
from google import genai
from IPython.display import Audio, display
from pydub import AudioSegment

def obtener_saludo_y_momento():
    """Calcula el saludo radiofónico dinámico según la hora actual."""
    hora = datetime.datetime.now().hour
    if 6 <= hora < 12:
        return "Buenos días"
    elif 12 <= hora < 20:
        return "Buenas tardes"
    else:
        return "Buenas noches"

def acelerar_audio(archivo_entrada, archivo_salida, velocidad=1.25):
    """Aumenta la velocidad del audio sin alterar el tono de la voz."""
    audio = AudioSegment.from_file(archivo_entrada)
    audio_rapido = audio._spawn(audio.raw_data, overrides={
        "frame_rate": int(audio.frame_rate * velocidad)
    })
    audio_rapido = audio_rapido.set_frame_rate(audio.frame_rate)
    audio_rapido.export(archivo_salida, format="mp3")

def generar_voz_espanol(texto, archivo_salida="boletin_trafico.mp3", velocidad=1.25):
    """Sintetiza el texto fragmentado y aplica la velocidad ajustada."""
    texto_limpio = re.sub(r'[*#\_]', '', texto)

    partes = re.split(r'(?<=[.?!])\s+', texto_limpio)
    fragmentos = []

    for parte in partes:
        if len(parte) > 180:
            fragmentos.extend(re.split(r'(?<=[,;])\s+', parte))
        else:
            fragmentos.append(parte)

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    bytes_audio_totales = bytearray()

    print("⏳ Descargando fragmentos de voz...")
    for fragmento in fragmentos:
        fragmento = fragmento.strip()
        if not fragmento:
            continue

        base_url = "https://translate.google.com/translate_tts"
        params = {"ie": "UTF-8", "q": fragmento, "tl": "es", "client": "tw-ob"}

        res = requests.get(base_url, params=params, headers=headers)
        if res.status_code == 200:
            bytes_audio_totales.extend(res.content)

    if bytes_audio_totales:
        temp_file = "temp_boletin.mp3"
        with open(temp_file, "wb") as f:
            f.write(bytes_audio_totales)

        acelerar_audio(temp_file, archivo_salida, velocidad=velocidad)
        return True
    return False

def generar_boletin_audio(velocidad=1.25):
    print("🔄 Extrayendo datos en tiempo real de la DGT...")
    datos_trafico = obtener_incidencias_texto()

    if "Sin incidencias" in datos_trafico or "Error" in datos_trafico:
        print(datos_trafico)
        return

    saludo_dinamico = obtener_saludo_y_momento()
    print(f"🤖 Generando boletín con Gemini 3.6 Flash ({saludo_dinamico})...\n")

    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)

    prompt = f"""
    Eres un locutor de radio experto en información de tráfico y tiempo regional. Genera un boletín locutado muy breve y fluido (110-130 palabras) para ser leído en voz alta sobre las incidencias en tiempo real de la DGT para Aragón, Navarra y La Rioja.

    REGLAS DE ESTRUCTURA Y FORMATO:
    1. **Saludo dinámico:** Comienza obligatoriamente con el saludo "{saludo_dinamico}" en lugar de un saludo fijo.
    2. **Apunte meteorológico rápido:** Tras el saludo, incluye una frase muy breve (10-15 palabras) sobre la situación del tiempo probable en el valle del Ebro y la zona norte (ejemplo: temperaturas, presencia de viento o posibles lluvias que puedan condicionar la conducción).
    3. **Estado del tráfico:** Resume de forma continua y radiofónica las incidencias más destacadas por regiones.
    4. **Sin marcas visuales:** NO uses emojis, asteriscos (*) ni encabezados markdown (##).
    5. **Nombres conocidos:** Asocia nombres populares a las carreteras (ej. "Autovía de Logroño", "Carretera de Belate", "Ronda de Zaragoza").

    DATOS DGT:
    {datos_trafico}
    """

    response = client.models.generate_content(
        model='gemini-3.6-flash',
        contents=prompt
    )

    texto_informe = response.text

    print("=" * 70)
    print("        TRANSCRIPCIÓN DEL BOLETÍN (CON TIEMPO Y SALUDO ADAPTADO)")
    print("=" * 70 + "\n")
    print(texto_informe)

    print(f"\n🔊 Generando audio acelerado ({velocidad}x) en castellano...")

    if generar_voz_espanol(texto_informe, "boletin_trafico.mp3", velocidad=velocidad):
        print("✅ ¡Audio MP3 adaptado generado con éxito!\n")
        display(Audio("boletin_trafico.mp3", autoplay=True))

print("✅ Generador de voz con tiempo y saludo adaptado configurado.")

✅ Generador de voz con tiempo y saludo adaptado configurado.


📢Procesar boletín de trafico

In [ ]:
# Ejecutar la prueba
generar_boletin_audio()

🔄 Extrayendo datos en tiempo real de la DGT...
🤖 Generando boletín con Gemini 3.6 Flash (Buenas noches)...

        TRANSCRIPCIÓN DEL BOLETÍN (CON TIEMPO Y SALUDO ADAPTADO)

Buenas noches. Cielos cubiertos y lloviznas dispersas que pueden complicar la visibilidad en el valle del Ebro. En Aragón, máxima atención en la provincia de Huesca, donde permanece cortada la N-330 a la altura de Canfranc y la A-132 entre Bailo y Las Peñas de Riglos. Además, en la Ronda de Zaragoza, la Z-40, tenemos un carril cortado junto a la Feria de Muestras, y persisten las obras en la Autovía de Logroño en El Burgo de Ebro. Nos vamos a Navarra, con un carril cerrado por trabajos en la Carretera de Belate, la N-121-A, en Anue. Finalmente, en La Rioja, mucha precaución por un accidente en la LR-341 a su paso por Ventosa y por desprendimientos en Villar de Torre. Prudencia al volante.

🔊 Generando audio acelerado (1.25x) en castellano...
⏳ Descargando fragmentos de voz...
✅ ¡Audio MP3 adaptado generado con éxit